# 04 — Synthetic ground-truth Yamada recovery

Controlled end-to-end validation:

$$
G_{\rm true}\rightarrow F(G_{\rm true})\rightarrow V_N
\rightarrow \widehat G\rightarrow \Upsilon(\widehat G;A).
$$

`F` is always an orientation-preserving invertible affine map, hence the
continuous spatial graph is ambient-isotopic to the ground truth before
voxelization. The benchmark is restricted to connected, bridgeless, exactly
trivalent graphs; closed knot/link components encoded as self-loops are excluded
because that representation currently produces projection pathologies under
some rotations.

`QUICK_MODE=True` is a small execution smoke test. Set it to `False` for the
requested $N=150,175,\ldots,300$ local sweep. Recovery mismatches are benchmark
results, not execution errors.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import hashlib, json, sys
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp
from skimage.morphology import ball, dilation, skeletonize

ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/"pyproject.toml").exists(): ROOT=ROOT.parent
SRC=ROOT/"src"
if not (SRC/"knotted_graph").exists(): raise RuntimeError("Run inside the KnottedGraph checkout.")
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))

import knotted_graph
from knotted_graph.core import simplify_edges
from knotted_graph.extraction import skeleton_image_to_graph
from knotted_graph.projection import compute_yamada_polynomial

kg_path=Path(knotted_graph.__file__).resolve()
if SRC not in kg_path.parents: raise RuntimeError(f"stale knotted_graph import: {kg_path}")

A=sp.Symbol("A"); BOUND=1.35; PROJECTION_SAMPLES=16
QUICK_MODE=True
if QUICK_MODE:
    RESOLUTIONS=[80]; TUBE_RADII_VOX=[1]; ACTIVE=["theta3_planar","K4","triangular_prism"]
else:
    RESOLUTIONS=list(range(150,301,25)); TUBE_RADII_VOX=[1,2,3]; ACTIVE=None
TRANSFORMS=["identity","rotate","affine"]
CACHE_SCHEMA="trivalent-ground-truth-v4"
CHECKPOINT=ROOT/"User_guide"/"benchmarks"/"synthetic_ground_truth_results_v4.jsonl"
RESUME=True
print("KnottedGraph:",kg_path)
print("mode:", "QUICK" if QUICK_MODE else "FULL", "resolutions:",RESOLUTIONS)


In [ ]:
@dataclass
class Case:
    name:str
    graph:nx.MultiGraph
    radius_cap:float

def emb(pos, edges):
    H=nx.MultiGraph()
    for n,p in pos.items(): H.add_node(n,pos=np.asarray(p,float))
    for u,v,P in edges: H.add_edge(u,v,pts=np.asarray(P,float))
    return H

def normalize(X,scale=.72):
    X=np.asarray(X,float); X-=X.mean(0)
    return X*(scale/np.max(np.linalg.norm(X,axis=1)))

def theta_case(name,bowed=False,n=500):
    t=np.linspace(0,1,n); x=-.72+1.44*t
    if bowed:
        C=[np.c_[x,-.58*np.sin(np.pi*t), .16*np.sin(2*np.pi*t)],
           np.c_[x, .10*np.sin(2*np.pi*t),-.10*np.sin(np.pi*t)],
           np.c_[x, .58*np.sin(np.pi*t),-.16*np.sin(2*np.pi*t)]]
    else:
        C=[np.c_[x,-.58*np.sin(np.pi*t),0*t],np.c_[x,0*t,0*t],np.c_[x,.58*np.sin(np.pi*t),0*t]]
    for P in C: P[0]=[-.72,0,0]; P[-1]=[.72,0,0]
    return Case(name,emb({"u":C[0][0],"v":C[0][-1]},[("u","v",P) for P in C]),.06)

def segdist(p1,q1,p2,q2):
    u=q1-p1; v=q2-p2; w=p1-p2
    a=u@u; b=u@v; c=v@v; d=u@w; e=v@w; D=a*c-b*b
    if D<1e-14: s=0.; t=np.clip(e/c if c>1e-14 else 0.,0,1)
    else: s=np.clip((b*e-c*d)/D,0,1); t=np.clip((a*e-b*d)/D,0,1)
    if a>1e-14: s=np.clip((b*t-d)/a,0,1)
    if c>1e-14: t=np.clip((b*s+e)/c,0,1)
    return float(np.linalg.norm(w+s*u-t*v))

def clearance(G,P):
    E=list(G.edges()); best=np.inf
    for i,(u,v) in enumerate(E):
        for a,b in E[i+1:]:
            if {u,v}&{a,b}: continue
            best=min(best,segdist(P[u],P[v],P[a],P[b]))
    return best

def cubic_case(name,G,planar,seed,cap=.04):
    G=nx.Graph(G)
    assert nx.is_connected(G) and all(d==3 for _,d in G.degree()) and not list(nx.bridges(G))
    if planar:
        ok,_=nx.check_planarity(G); assert ok
        p=nx.planar_layout(G); X=normalize([[p[n][0],p[n][1],0.] for n in G])
        P={n:X[i] for i,n in enumerate(G)}
    else:
        P=None
        for trial in range(200):
            p=nx.spring_layout(G,dim=3,seed=seed+trial,iterations=500)
            X=normalize([p[n] for n in G]); Q={n:X[i] for i,n in enumerate(G)}
            if clearance(G,Q)>.055: P=Q; break
        if P is None: raise RuntimeError(f"no clear embedding for {name}")
    return Case(name,emb(P,[(u,v,np.linspace(P[u],P[v],80)) for u,v in G.edges()]),cap)

ALL_CASES=[
 theta_case("theta3_planar"),theta_case("theta3_bowed",True),
 cubic_case("K4",nx.complete_graph(4),True,11,.052),
 cubic_case("triangular_prism",nx.circular_ladder_graph(3),True,12,.045),
 cubic_case("cube",nx.cubical_graph(),True,13,.042),
 cubic_case("pentagonal_prism",nx.circular_ladder_graph(5),True,14,.035),
 cubic_case("dodecahedral",nx.dodecahedral_graph(),True,15,.027),
 cubic_case("K3_3",nx.complete_bipartite_graph(3,3),False,17,.032),
 cubic_case("petersen",nx.petersen_graph(),False,18,.030),
 cubic_case("heawood",nx.heawood_graph(),False,19,.024),
]
by_name={c.name:c for c in ALL_CASES}
CASES=ALL_CASES if ACTIVE is None else [by_name[n] for n in ACTIVE]
for c in ALL_CASES:
    assert nx.is_connected(nx.Graph(c.graph))
    assert all(d==3 for d in dict(c.graph.degree()).values())
print([(c.name,c.graph.number_of_nodes(),c.graph.number_of_edges()) for c in CASES])


In [ ]:
def Rxyz(a,b,c):
    a,b,c=np.deg2rad([a,b,c])
    Rx=np.array([[1,0,0],[0,np.cos(a),-np.sin(a)],[0,np.sin(a),np.cos(a)]])
    Ry=np.array([[np.cos(b),0,np.sin(b)],[0,1,0],[-np.sin(b),0,np.cos(b)]])
    Rz=np.array([[np.cos(c),-np.sin(c),0],[np.sin(c),np.cos(c),0],[0,0,1]])
    return Rz@Ry@Rx

def transform(name):
    if name=="identity": M,b=np.eye(3),np.zeros(3)
    elif name=="rotate": M,b=Rxyz(21,34,13),np.array([.04,-.03,.02])
    else:
        M=Rxyz(17,-23,31)@np.diag([1.08,.91,1.03])@np.array([[1,.13,0],[0,1,.09],[.05,0,1]])
        b=np.array([-.03,.04,-.02])
    assert np.linalg.det(M)>0
    return M,b

def deform(G,name):
    M,b=transform(name); H=nx.MultiGraph()
    for n,d in G.nodes(data=True): H.add_node(n,pos=d["pos"]@M.T+b)
    for u,v,k,d in G.edges(keys=True,data=True): H.add_edge(u,v,pts=d["pts"]@M.T+b)
    return H

def trimmed(P,f=.15):
    n=max(1,int(round(f*len(P))))
    return P[n:-n] if 2*n<len(P) else P

def interior_sep(G):
    E=[(u,v,np.asarray(d["pts"],float)) for u,v,k,d in G.edges(keys=True,data=True)]
    best=np.inf
    for i,(u,v,P0) in enumerate(E):
        for a,b,Q0 in E[i+1:]:
            P,Q=(trimmed(P0),trimmed(Q0)) if {u,v}&{a,b} else (P0,Q0)
            for s in range(0,len(P),128):
                best=min(best,float(np.sqrt(np.sum((P[s:s+128,None]-Q[None])**2,-1).min())))
    return best

def admissible(case,G,N,r):
    dx=2*BOUND/(N-1); rw=r*dx; sep=interior_sep(G)
    limit=min(case.radius_cap,.4*sep)
    return rw<=limit,rw,sep,limit

def resample(P,step):
    parts=[]
    for p,q in zip(P[:-1],P[1:]):
        n=max(2,int(np.ceil(np.linalg.norm(q-p)/step))+1)
        parts.append(np.linspace(p,q,n,endpoint=False))
    parts.append(P[-1:])
    return np.vstack(parts)

def voxelize(G,N,r):
    V=np.zeros((N,N,N),bool); dx=2*BOUND/(N-1)
    for _,_,_,d in G.edges(keys=True,data=True):
        P=resample(d["pts"],dx/3)
        I=np.rint((P+BOUND)/(2*BOUND)*(N-1)).astype(int); I=np.clip(I,0,N-1)
        V[I[:,0],I[:,1],I[:,2]]=1
    return dilation(V,footprint=ball(r))

def recover(V,N):
    raw=skeleton_image_to_graph(skeletonize(V,method="lee"))
    H=nx.MultiGraph(raw); dx=2*BOUND/(N-1); o=np.array([-BOUND]*3)
    for _,d in H.nodes(data=True): d["pos"]=o+dx*np.asarray(d["pos"],float)
    for _,_,_,d in H.edges(keys=True,data=True): d["pts"]=o+dx*np.asarray(d["pts"],float)
    return simplify_edges(H)

def yamada(G):
    r=compute_yamada_polynomial(G,A,num_rotation_samples=PROJECTION_SAMPLES,
        crossing_warning_threshold=None,normalize=True,n_jobs=1,method="recursive",return_result=True)
    return sp.expand(r.polynomial),r.projection

def same(a,b): return sp.simplify(sp.together(sp.expand(a-b)))==0


In [ ]:
TARGETS={}
for c in CASES:
    TARGETS[c.name],p=yamada(c.graph)
    print(f"TARGET {c.name:20s} x={p.num_crossings:2d} Yamada={TARGETS[c.name]}")

for c in CASES:
    for t in TRANSFORMS:
        poly,p=yamada(deform(c.graph,t))
        if not same(poly,TARGETS[c.name]):
            raise AssertionError(
                f"{c.name}/{t}: graph-level invariance failed before voxelization; "
                f"target={TARGETS[c.name]}, got={poly}, crossings={p.num_crossings}"
            )
print("PASS: all active affine deformations preserve graph-level Yamada.")


In [ ]:
payload={"schema":CACHE_SCHEMA,"mode":"quick" if QUICK_MODE else "full","cases":[c.name for c in CASES],
         "N":RESOLUTIONS,"r":TUBE_RADII_VOX,"transforms":TRANSFORMS}
SIGNATURE=hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()[:20]
def key(c,t,N,r): return (SIGNATURE,c,t,int(N),int(r))

done={}
if RESUME and CHECKPOINT.exists():
    for line in CHECKPOINT.read_text().splitlines():
        if line.strip():
            row=json.loads(line)
            if row.get("signature")==SIGNATURE:
                done[key(row["case"],row["transform"],row["resolution"],row["radius_vox"])]=row
records=[]; CHECKPOINT.parent.mkdir(parents=True,exist_ok=True)

for c in CASES:
  for t in TRANSFORMS:
    G=deform(c.graph,t)
    for N in RESOLUTIONS:
      for r in TUBE_RADII_VOX:
        K=key(c.name,t,N,r)
        if K in done: records.append(done[K]); print("CACHED",K[1:]); continue
        ok,rw,sep,lim=admissible(c,G,N,r)
        row={"signature":SIGNATURE,"case":c.name,"transform":t,"resolution":N,"radius_vox":r,
             "radius_world":rw,"separation":sep,"limit":lim,"admissible":bool(ok),
             "success":None,"final_V":None,"final_E":None,"selected_crossings":None,
             "recovered_yamada":None,"error":None}
        if ok:
            try:
                H=recover(voxelize(G,N,r),N); poly,p=yamada(H)
                row.update(success=bool(same(poly,TARGETS[c.name])),final_V=H.number_of_nodes(),
                           final_E=H.number_of_edges(),selected_crossings=p.num_crossings,
                           recovered_yamada=str(poly))
            except Exception as exc:
                row.update(success=False,error=f"{type(exc).__name__}: {exc}")
        records.append(row)
        with CHECKPOINT.open("a") as f: f.write(json.dumps(row)+"\n")
        mark="SKIP" if not ok else ("PASS" if row["success"] else "FAIL")
        print(f"{mark:4s} {c.name:20s} {t:8s} N={N} r={r} V/E={row['final_V']}/{row['final_E']}")

valid=[x for x in records if x["admissible"]]
if not valid: raise RuntimeError("No admissible parameter point survived the tube-thickness guard.")
passed=sum(bool(x["success"]) for x in valid)
print(f"\nHeadline Yamada recovery: {passed}/{len(valid)} = {100*passed/len(valid):.2f}%")
for x in valid:
    if not x["success"]:
        print("FAILURE:",x["case"],x["transform"],"N=",x["resolution"],"r=",x["radius_vox"],
              x["error"] or f"Yamada mismatch: {x['recovered_yamada']}")

if QUICK_MODE:
    errors=[x for x in valid if x["error"]]
    if errors: raise RuntimeError(f"Quick smoke test had execution errors: {errors}")
    baseline=[x for x in valid if x["case"]=="theta3_planar" and x["transform"]=="identity"]
    if not baseline or not all(x["success"] for x in baseline):
        raise AssertionError("Known theta3_planar/identity recovery baseline failed.")
    print("PASS: quick execution smoke test completed; recovery mismatches above are measured benchmark outcomes.")


In [ ]:
H=np.full((len(TUBE_RADII_VOX),len(RESOLUTIONS)),np.nan)
for i,r in enumerate(TUBE_RADII_VOX):
    for j,N in enumerate(RESOLUTIONS):
        g=[x for x in records if x["admissible"] and x["radius_vox"]==r and x["resolution"]==N]
        if g: H[i,j]=np.mean([bool(x["success"]) for x in g])
fig,ax=plt.subplots(figsize=(8,3.5)); im=ax.imshow(H,vmin=0,vmax=1,origin="lower",aspect="auto")
ax.set_xticks(range(len(RESOLUTIONS)),RESOLUTIONS); ax.set_yticks(range(len(TUBE_RADII_VOX)),TUBE_RADII_VOX)
ax.set_xlabel("voxel resolution N"); ax.set_ylabel("tube radius [voxels]"); ax.set_title("Yamada recovery rate")
fig.colorbar(im,ax=ax,label="recovery fraction"); plt.show()

names=[c.name for c in CASES]
rates=[]
for n in names:
    g=[x for x in records if x["admissible"] and x["case"]==n]
    rates.append(np.mean([bool(x["success"]) for x in g]) if g else np.nan)
fig,ax=plt.subplots(figsize=(10,4)); ax.bar(range(len(names)),rates); ax.set_ylim(0,1.05)
ax.set_xticks(range(len(names)),names,rotation=60,ha="right"); ax.set_ylabel("Yamada recovery fraction")
plt.tight_layout(); plt.show()


### Interpretation

A mismatch after voxelization is a **measured recovery failure**, not a notebook
execution failure. This is precisely what the resolution/thickness validity map
is intended to expose. `admissible=False` combinations are excluded because the
continuous tube is already too thick relative to geometric clearance.

Before a full local run, execute the notebook once with `QUICK_MODE=True`. It
must finish and print the quick-smoke PASS line. Then set `QUICK_MODE=False` and
rerun for $N=150,\ldots,300$. The versioned checkpoint makes the long run
resumable.
